# 四模型（TFT / LSTM / GRU / Vanilla Transformer）超參數搜尋 —— Optuna

請在本機 TFT_env（已安裝 torch / pytorch_forecasting / lightning）逐格執行，另需 `pip install optuna`。

**方法：**
- TFT：使用 pytorch_forecasting 內建的 `optimize_hyperparameters()`（底層即 Optuna），不需要自己寫 objective 函式。
- LSTM / GRU（`RecurrentNetwork`）與自訂的 Vanilla Transformer：沒有現成調參函式，故自行寫 Optuna 的 `objective(trial)` 函式 + 自訂 pruning callback（不依賴 `optuna-integration` 套件，避免版本相容性問題）。

四模型共用相同的搜尋預算（`N_TRIALS`、`MAX_EPOCHS_PER_TRIAL`）與共通超參數的搜尋範圍（`hidden_size`、`dropout`、`learning_rate`），以確保跨模型比較的公平性。

**輸出：**
- `optuna_best_hyperparameters.json` —— 四模型找到的最佳超參數
- `optuna_trials_TFT.csv` / `_LSTM.csv` / `_GRU.csv` / `_Transformer.csv` —— 各模型完整 trial 紀錄
- `optuna_收斂圖_{模型}.png` —— 各模型的優化收斂曲線（可放論文附錄）

找到最佳超參數後，請套用回你原本 `TFT.ipynb` / `baseline_models.ipynb` 的正式訓練流程（完整 epoch 數 + EarlyStopping），重新訓練最終模型。

## 1. 匯入套件與路徑設定

請把 `BASE_DIR` 改成你「能源」資料夾的實際絕對路徑。

In [ ]:
!pip install optuna
!pip install optuna statsmodels optuna-integration

In [ ]:
import json
import math
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import lightning.pytorch as pl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

# ======================= 路徑設定，請依實際情況修改 =======================
import os

BASE_DIR = r"C:\Users\peggy\Desktop\能源"  # 請改成你「能源」資料夾的實際路徑

# 切換工作目錄到BASE_DIR，避免lightning_logs等相對路徑輸出因路徑過長在Windows報錯
os.chdir(BASE_DIR)
print("目前工作目錄：", os.getcwd())

DATASET_DIR = os.path.join(BASE_DIR, "dataset")
OUTPUT_DIR = BASE_DIR

MAX_ENCODER_LENGTH = 168
MAX_PREDICTION_LENGTH = 24
CAT_COLS = ["building_id", "site_id", "hour", "weekday", "is_weekend", "month"]

## 2. 搜尋預算與共用超參數範圍

`N_TRIALS`／`MAX_EPOCHS_PER_TRIAL` 可依運算資源調整；四模型共用同一組範圍與預算以確保公平比較。

In [ ]:
# ======================= 搜尋預算與共用超參數範圍（四模型一致，確保公平比較）=======================
N_TRIALS = 20              # 每個模型跑幾次trial，可依運算資源調整
MAX_EPOCHS_PER_TRIAL = 6   # 搜尋階段每個trial的epoch數（找到最佳參數後再用完整epoch數重新訓練）
RANDOM_SEED = 42

HIDDEN_SIZE_RANGE = (8, 128)       # LSTM/GRU/Transformer的d_model相關搜尋會另外處理類別型範圍
DROPOUT_RANGE = (0.0, 0.3)
LEARNING_RATE_RANGE = (1e-4, 1e-1)  # log-uniform

pl.seed_everything(RANDOM_SEED)

## 3. 中文字型設定（matplotlib）

In [ ]:
# ======================= 中文字型（matplotlib）=======================
def setup_cjk_font():
    candidates = [
        "Microsoft JhengHei", "Microsoft YaHei", "SimHei",
        "Noto Sans CJK TC", "Noto Sans CJK SC", "PingFang TC",
    ]
    available = {f.name for f in fm.fontManager.ttflist}
    for name in candidates:
        if name in available:
            plt.rcParams["font.sans-serif"] = [name]
            plt.rcParams["axes.unicode_minus"] = False
            print(f"使用中文字型: {name}")
            return
    print("警告：找不到常見中文字型，圖片中的中文可能無法正常顯示")


## 4. 共用函式：讀資料、建立 train/val dataset 與 dataloader

In [ ]:
# ======================= 共用：讀資料 + 建立 train/val dataset =======================
def load_seen_full(fix_zero_bug: bool):
    """fix_zero_bug=True 對應 baseline_models.ipynb 的做法（LSTM/GRU/Transformer 使用）；
    TFT 原始流程未套用此修正，故TFT搜尋請維持 fix_zero_bug=False，
    以符合最終要重新訓練之TFT正式流程的資料分布。"""
    train = pd.read_parquet(f"{DATASET_DIR}/train.parquet")
    val = pd.read_parquet(f"{DATASET_DIR}/val.parquet")
    test = pd.read_parquet(f"{DATASET_DIR}/test.parquet")
    seen_full = pd.concat([train, val, test], ignore_index=True)
    seen_full["timestamp"] = pd.to_datetime(seen_full["timestamp"])
    seen_full = seen_full.sort_values(["building_id", "timestamp"]).reset_index(drop=True)
    seen_full["time_idx"] = (
        (seen_full["timestamp"] - seen_full["timestamp"].min()).dt.total_seconds().div(3600).astype(int)
    )
    if fix_zero_bug:
        seen_full.loc[seen_full["load_intensity"] == 0.0, "load_intensity"] = np.nan
        seen_full["load_intensity"] = (
            seen_full.groupby("building_id")["load_intensity"]
            .transform(lambda s: s.interpolate(method="linear").ffill().bfill())
        )
    for col in CAT_COLS:
        seen_full[col] = seen_full[col].astype(str)
    return seen_full


def build_training_dataset(seen_full):
    from pytorch_forecasting import TimeSeriesDataSet
    from pytorch_forecasting.data import GroupNormalizer

    training_cutoff = seen_full[seen_full["timestamp"] < "2017-06-01"]["time_idx"].max()
    return TimeSeriesDataSet(
        seen_full[lambda x: x.time_idx <= training_cutoff],
        time_idx="time_idx",
        target="load_intensity",
        group_ids=["building_id"],
        min_encoder_length=MAX_ENCODER_LENGTH // 2,
        max_encoder_length=MAX_ENCODER_LENGTH,
        min_prediction_length=1,
        max_prediction_length=MAX_PREDICTION_LENGTH,
        static_categoricals=["building_id", "site_id"],
        static_reals=["sqm"],
        time_varying_known_categoricals=["hour", "weekday", "is_weekend", "month"],
        time_varying_known_reals=["time_idx", "airTemperature", "dewTemperature", "windSpeed", "windDirection"],
        time_varying_unknown_reals=["load_intensity"],
        target_normalizer=GroupNormalizer(groups=["building_id"], transformation="softplus"),
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
    )


def build_train_val_dataloaders(seen_full, training_dataset, batch_size_train=512, batch_size_val=1024):
    from pytorch_forecasting import TimeSeriesDataSet

    validation_dataset = TimeSeriesDataSet.from_dataset(
        training_dataset, seen_full, predict=True, stop_randomization=True
    )
    # num_workers=0：避免Windows上多進程DataLoader worker在反覆多次trial訓練時崩潰
    # （之前在正式訓練TFT時遇過 "DataLoader worker exited unexpectedly"，搜尋階段trial更多次，更容易踩到）
    train_dataloader = training_dataset.to_dataloader(train=True, batch_size=batch_size_train, num_workers=0)
    val_dataloader = validation_dataset.to_dataloader(train=False, batch_size=batch_size_val, num_workers=0)
    return train_dataloader, val_dataloader


# ---- 準備 LSTM / GRU / Transformer 共用資料（fix_zero_bug=True）----
print("\n===== 準備 LSTM / GRU / Transformer 共用資料 =====")
seen_full_fixed = load_seen_full(fix_zero_bug=True)
training_dataset_fixed = build_training_dataset(seen_full_fixed)
train_dataloader_fixed, val_dataloader_fixed = build_train_val_dataloaders(seen_full_fixed, training_dataset_fixed)

# ---- 準備 TFT 專用資料（fix_zero_bug=False，維持TFT原始流程）----
print("\n===== 準備 TFT 專用資料 =====")
seen_full_raw = load_seen_full(fix_zero_bug=False)
training_dataset_raw = build_training_dataset(seen_full_raw)
train_dataloader_raw, val_dataloader_raw = build_train_val_dataloaders(seen_full_raw, training_dataset_raw)

## 5. 自訂 Optuna pruning callback

不依賴 `optuna-integration` 套件（版本相容性問題），自己寫一個最小可用版本，只依賴 optuna 本身。

In [ ]:
# ======================= 自訂 Optuna pruning callback =======================
# 不依賴 optuna-integration 套件（該套件的PyTorchLightningPruningCallback在不同optuna版本間API常變動，
# 曾經在這個專案裡吃過版本相容性的虧，故自己寫一個最小可用版本，只依賴optuna本身）
class OptunaPruningCallback(pl.Callback):
    def __init__(self, trial, monitor="val_loss"):
        super().__init__()
        self.trial = trial
        self.monitor = monitor

    def on_validation_end(self, trainer, pl_module):
        current_score = trainer.callback_metrics.get(self.monitor)
        if current_score is None:
            return
        self.trial.report(current_score.item(), step=trainer.current_epoch)
        if self.trial.should_prune():
            raise optuna.TrialPruned()


def make_trainer(trial, accelerator):
    return pl.Trainer(
        max_epochs=MAX_EPOCHS_PER_TRIAL,
        accelerator=accelerator,
        devices=1,
        enable_progress_bar=False,
        enable_model_summary=False,
        logger=False,
        gradient_clip_val=0.1,
        callbacks=[OptunaPruningCallback(trial, monitor="val_loss")],
    )


ACCELERATOR = "gpu" if torch.cuda.is_available() else "cpu"
print("\n使用裝置：", ACCELERATOR)

all_best_params = {}
all_studies = {}

## 6. TFT 超參數搜尋

使用 pytorch_forecasting 內建的 `optimize_hyperparameters()`。若跑此格出現 TypeError 提示 unexpected keyword，請執行 `help(optimize_hyperparameters)` 確認實際參數名稱後調整（不同版本的 pytorch_forecasting 參數名稱可能略有差異）。

In [ ]:
# ======================= TFT：使用pytorch_forecasting內建的optimize_hyperparameters =======================
print("\n===== TFT 超參數搜尋（使用pytorch_forecasting內建optimize_hyperparameters）=====")
from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters

# 注意：不同版本的pytorch_forecasting，這個函式的參數名稱可能略有差異。
# 若跑此格出現TypeError提示unexpected keyword，請執行 help(optimize_hyperparameters) 確認實際參數名稱後調整。
study_tft = optimize_hyperparameters(
    train_dataloader_raw,
    val_dataloader_raw,
    model_path=os.path.join(OUTPUT_DIR, "optuna_tft_tmp"),
    n_trials=N_TRIALS,
    max_epochs=MAX_EPOCHS_PER_TRIAL,
    gradient_clip_val_range=(0.01, 1.0),
    hidden_size_range=HIDDEN_SIZE_RANGE,
    hidden_continuous_size_range=(8, 64),
    attention_head_size_range=(1, 4),
    learning_rate_range=LEARNING_RATE_RANGE,
    dropout_range=DROPOUT_RANGE,
    trainer_kwargs=dict(
        accelerator=ACCELERATOR, devices=1, enable_progress_bar=False,
        enable_model_summary=False, logger=True,
    ),
    reduce_on_plateau_patience=4,
    use_learning_rate_finder=False,
)

print("TFT 最佳超參數：", study_tft.best_trial.params)
all_best_params["TFT"] = study_tft.best_trial.params
all_studies["TFT"] = study_tft

## 7. LSTM / GRU 超參數搜尋（自訂 objective）

In [ ]:
# ======================= LSTM / GRU：自訂 objective =======================
from pytorch_forecasting import RecurrentNetwork
from pytorch_forecasting.metrics import RMSE


def make_recurrent_objective(cell_type):
    def objective(trial):
        hidden_size = trial.suggest_int("hidden_size", *HIDDEN_SIZE_RANGE, log=True)
        rnn_layers = trial.suggest_int("rnn_layers", 1, 3)
        dropout = trial.suggest_float("dropout", *DROPOUT_RANGE)
        learning_rate = trial.suggest_float("learning_rate", *LEARNING_RATE_RANGE, log=True)

        model = RecurrentNetwork.from_dataset(
            training_dataset_fixed,
            cell_type=cell_type,
            hidden_size=hidden_size,
            rnn_layers=rnn_layers,
            dropout=dropout,
            loss=RMSE(),
            learning_rate=learning_rate,
        )

        trainer = make_trainer(trial, ACCELERATOR)
        trainer.fit(model, train_dataloaders=train_dataloader_fixed, val_dataloaders=val_dataloader_fixed)
        return trainer.callback_metrics["val_loss"].item()

    return objective


print("\n===== LSTM 超參數搜尋 =====")
study_lstm = optuna.create_study(direction="minimize", sampler=TPESampler(seed=RANDOM_SEED), pruner=MedianPruner())
study_lstm.optimize(make_recurrent_objective("LSTM"), n_trials=N_TRIALS)
print("LSTM 最佳超參數：", study_lstm.best_trial.params)
all_best_params["LSTM"] = study_lstm.best_trial.params
all_studies["LSTM"] = study_lstm

print("\n===== GRU 超參數搜尋 =====")
study_gru = optuna.create_study(direction="minimize", sampler=TPESampler(seed=RANDOM_SEED), pruner=MedianPruner())
study_gru.optimize(make_recurrent_objective("GRU"), n_trials=N_TRIALS)
print("GRU 最佳超參數：", study_gru.best_trial.params)
all_best_params["GRU"] = study_gru.best_trial.params
all_studies["GRU"] = study_gru

## 8. Vanilla Transformer 超參數搜尋（自訂模型 + 自訂 objective）

In [ ]:
# ======================= Vanilla Transformer：自訂模型 + 自訂 objective =======================
print("\n===== 準備 Vanilla Transformer 模型類別 =====")

cat_names = ["building_id", "site_id", "hour", "weekday", "is_weekend", "month"]
cat_sizes = [len(training_dataset_fixed._categorical_encoders[name].classes_) for name in cat_names]
n_continuous = len(training_dataset_fixed.reals)
print("cat_sizes:", cat_sizes, " n_continuous:", n_continuous)


class VanillaTransformerNet(nn.Module):
    def __init__(self, cat_sizes, n_continuous, d_model=64, nhead=4,
                 num_layers=2, dim_feedforward=128, dropout=0.1, prediction_length=24):
        super().__init__()
        emb_dims = [min(50, (s + 1) // 2) for s in cat_sizes]
        self.embeddings = nn.ModuleList([nn.Embedding(s, d) for s, d in zip(cat_sizes, emb_dims)])

        total_input_dim = sum(emb_dims) + n_continuous
        self.input_projection = nn.Linear(total_input_dim, d_model)

        pe = torch.zeros(500, d_model)
        position = torch.arange(500).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pos_encoding", pe)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_projection = nn.Linear(d_model, prediction_length)

    def forward(self, encoder_cat, encoder_cont):
        embedded = [emb(encoder_cat[..., i]) for i, emb in enumerate(self.embeddings)]
        embedded = torch.cat(embedded, dim=-1)
        combined = torch.cat([embedded, encoder_cont], dim=-1)
        projected = self.input_projection(combined)
        seq_len = projected.size(1)
        projected = projected + self.pos_encoding[:seq_len].unsqueeze(0)
        encoded = self.encoder(projected)
        return self.output_projection(encoded[:, -1, :])


class VanillaTransformerModel(pl.LightningModule):
    def __init__(self, cat_sizes, n_continuous, d_model=64, nhead=4,
                 num_layers=2, dim_feedforward=128, dropout=0.1,
                 prediction_length=24, learning_rate=0.001):
        super().__init__()
        self.save_hyperparameters()
        self.net = VanillaTransformerNet(
            cat_sizes=cat_sizes, n_continuous=n_continuous, d_model=d_model,
            nhead=nhead, num_layers=num_layers, dim_feedforward=dim_feedforward,
            dropout=dropout, prediction_length=prediction_length,
        )
        self.learning_rate = learning_rate

    def forward(self, x):
        return self.net(x["encoder_cat"], x["encoder_cont"])

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = torch.sqrt(nn.functional.mse_loss(y_hat, y[0]) + 1e-8)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = torch.sqrt(nn.functional.mse_loss(y_hat, y[0]) + 1e-8)
        self.log("val_loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)


NHEAD_CHOICES = [2, 4, 8]
D_MODEL_CHOICES = [32, 64, 128]
DIM_FEEDFORWARD_CHOICES = [64, 128, 256]


def transformer_objective(trial):
    d_model = trial.suggest_categorical("d_model", D_MODEL_CHOICES)
    nhead = trial.suggest_categorical("nhead", NHEAD_CHOICES)
    if d_model % nhead != 0:
        # d_model必須能被nhead整除，不符合的組合直接視為此trial不可行
        raise optuna.TrialPruned()
    num_layers = trial.suggest_int("num_layers", 1, 4)
    dim_feedforward = trial.suggest_categorical("dim_feedforward", DIM_FEEDFORWARD_CHOICES)
    dropout = trial.suggest_float("dropout", *DROPOUT_RANGE)
    learning_rate = trial.suggest_float("learning_rate", *LEARNING_RATE_RANGE, log=True)

    model = VanillaTransformerModel(
        cat_sizes=cat_sizes, n_continuous=n_continuous,
        d_model=d_model, nhead=nhead, num_layers=num_layers,
        dim_feedforward=dim_feedforward, dropout=dropout,
        prediction_length=MAX_PREDICTION_LENGTH, learning_rate=learning_rate,
    )

    trainer = make_trainer(trial, ACCELERATOR)
    trainer.fit(model, train_dataloaders=train_dataloader_fixed, val_dataloaders=val_dataloader_fixed)
    return trainer.callback_metrics["val_loss"].item()


print("\n===== Vanilla Transformer 超參數搜尋 =====")
study_transformer = optuna.create_study(direction="minimize", sampler=TPESampler(seed=RANDOM_SEED), pruner=MedianPruner())
study_transformer.optimize(transformer_objective, n_trials=N_TRIALS)
print("Vanilla Transformer 最佳超參數：", study_transformer.best_trial.params)
all_best_params["Vanilla Transformer"] = study_transformer.best_trial.params
all_studies["Vanilla Transformer"] = study_transformer

## 9. 彙整輸出：最佳超參數 JSON + 各模型 trial 紀錄 CSV

In [ ]:
# ======================= 彙整輸出 =======================
with open(os.path.join(OUTPUT_DIR, "optuna_best_hyperparameters.json"), "w", encoding="utf-8") as f:
    json.dump(all_best_params, f, ensure_ascii=False, indent=2)
print("\n已輸出 optuna_best_hyperparameters.json：")
print(json.dumps(all_best_params, ensure_ascii=False, indent=2))

for model_name, study in all_studies.items():
    df = study.trials_dataframe()
    csv_name = f"optuna_trials_{model_name.replace(' ', '_')}.csv"
    df.to_csv(os.path.join(OUTPUT_DIR, csv_name), index=False, encoding="utf-8-sig")
    print(f"已輸出 {csv_name}（共{len(df)}筆trial紀錄）")

## 10. 繪圖：各模型優化收斂曲線

In [ ]:
# ======================= 繪圖：各模型優化收斂曲線 =======================
for model_name, study in all_studies.items():
    df = study.trials_dataframe()
    completed = df[df["state"] == "COMPLETE"].sort_values("number")
    if len(completed) == 0:
        print(f"{model_name}：沒有成功完成的trial，略過繪圖")
        continue
    running_best = completed["value"].cummin()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(completed["number"], completed["value"], alpha=0.5, label="每次trial的val_loss", color="#4C72B0")
    ax.plot(completed["number"], running_best, color="#C44E52", linewidth=2, label="目前最佳值")
    ax.set_xlabel("Trial 編號")
    ax.set_ylabel("val_loss")
    ax.set_title(f"{model_name} 超參數搜尋收斂曲線（共{N_TRIALS}次trial）")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fname = f"optuna_收斂圖_{model_name.replace(' ', '_')}.png"
    plt.savefig(os.path.join(OUTPUT_DIR, fname), dpi=150)
    plt.close()
    print(f"已輸出 {fname}")

print("\n找到最佳超參數！接下來用這組參數＋完整epoch數＋EarlyStopping重新訓練最終模型，並記錄逐epoch的train_loss/val_loss。")

## 11. 用最佳超參數重新訓練最終模型，並記錄逐epoch損失曲線

用Optuna找到的最佳超參數，套用完整epoch數（`FINAL_MAX_EPOCHS=30`）+ EarlyStopping重新訓練四個最終模型（沿用原本TFT.ipynb／baseline_models.ipynb的正式訓練設定），並用自訂的`EpochLossRecorder`記錄逐epoch的train_loss/val_loss，訓練完直接存成新的checkpoint。

In [ ]:
# ======================= 用最佳超參數重新訓練最終模型，並記錄逐epoch損失曲線 =======================
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

FINAL_MAX_EPOCHS = 30  # 與原本TFT.ipynb／baseline_models.ipynb的正式訓練epoch數一致


class EpochLossRecorder(pl.Callback):
    """記錄每個epoch結束時的train_loss/val_loss，供事後畫損失曲線用。"""
    def __init__(self):
        super().__init__()
        self.train_losses = []
        self.val_losses = []

    def on_train_epoch_end(self, trainer, pl_module):
        loss = trainer.callback_metrics.get("train_loss")
        if loss is not None:
            self.train_losses.append(loss.item())

    def on_validation_epoch_end(self, trainer, pl_module):
        loss = trainer.callback_metrics.get("val_loss")
        if loss is not None:
            self.val_losses.append(loss.item())


def make_early_stop():
    return EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=3, verbose=True, mode="min")


final_recorders = {}
final_models = {}

# ---- TFT：用最佳超參數重新訓練 ----
print("\n===== 用最佳超參數重新訓練 TFT =====")
from pytorch_forecasting.metrics import QuantileLoss

tft_params = dict(all_best_params["TFT"])
gradient_clip_val_tft = tft_params.pop("gradient_clip_val", 0.1)

tft_final = TemporalFusionTransformer.from_dataset(
    training_dataset_raw,
    loss=QuantileLoss(quantiles=[0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]),
    log_interval=10,
    optimizer="Adam",
    reduce_on_plateau_patience=4,
    **tft_params,
)

recorder_tft = EpochLossRecorder()
checkpoint_tft = ModelCheckpoint(
    dirpath="tft_checkpoints", filename="best-tft-optuna-{epoch:02d}-{val_loss:.4f}",
    monitor="val_loss", mode="min", save_top_k=1,
)
trainer_tft = pl.Trainer(
    max_epochs=FINAL_MAX_EPOCHS,
    accelerator=ACCELERATOR, devices=1,
    gradient_clip_val=gradient_clip_val_tft,
    precision="32",  # 避免bf16/fp16在此模型上曾遇過的數值問題
    callbacks=[make_early_stop(), checkpoint_tft, recorder_tft],
    logger=False,
    enable_progress_bar=True,
)

tft_final.train()  # 確保模型處於train模式（避免cudnn RNN backward在eval模式下報錯）
trainer_tft.fit(tft_final, train_dataloaders=train_dataloader_raw, val_dataloaders=val_dataloader_raw)
final_recorders["TFT"] = recorder_tft
final_models["TFT"] = tft_final
print("TFT 最佳模型儲存位置：", checkpoint_tft.best_model_path)

# ---- LSTM / GRU：用最佳超參數重新訓練 ----
for cell_type in ["LSTM", "GRU"]:
    print(f"\n===== 用最佳超參數重新訓練 {cell_type} =====")
    model_params = dict(all_best_params[cell_type])
    model_final = RecurrentNetwork.from_dataset(
        training_dataset_fixed,
        cell_type=cell_type,
        loss=RMSE(),
        **model_params,
    )
    recorder = EpochLossRecorder()
    checkpoint_cb = ModelCheckpoint(
        dirpath=f"{cell_type.lower()}_checkpoints", filename=f"best-{cell_type.lower()}-optuna-{{epoch:02d}}-{{val_loss:.4f}}",
        monitor="val_loss", mode="min", save_top_k=1,
    )
    trainer_final = pl.Trainer(
        max_epochs=FINAL_MAX_EPOCHS,
        accelerator=ACCELERATOR, devices=1,
        gradient_clip_val=0.1,
        callbacks=[make_early_stop(), checkpoint_cb, recorder],
        logger=False,
        enable_progress_bar=True,
    )
    trainer_final.fit(model_final, train_dataloaders=train_dataloader_fixed, val_dataloaders=val_dataloader_fixed)
    final_recorders[cell_type] = recorder
    final_models[cell_type] = model_final
    print(f"{cell_type} 最佳模型儲存位置：", checkpoint_cb.best_model_path)

# ---- Vanilla Transformer：用最佳超參數重新訓練 ----
print("\n===== 用最佳超參數重新訓練 Vanilla Transformer =====")


class VanillaTransformerModelFinal(pl.LightningModule):
    """與搜尋階段的VanillaTransformerModel完全相同，唯一差異是self.log()明確加上on_epoch=True，
    確保EpochLossRecorder能穩定抓到逐epoch的平均損失（純粹是記錄用途的差異，不影響模型本身行為）。"""
    def __init__(self, cat_sizes, n_continuous, d_model=64, nhead=4,
                 num_layers=2, dim_feedforward=128, dropout=0.1,
                 prediction_length=24, learning_rate=0.001):
        super().__init__()
        self.save_hyperparameters()
        self.net = VanillaTransformerNet(
            cat_sizes=cat_sizes, n_continuous=n_continuous, d_model=d_model,
            nhead=nhead, num_layers=num_layers, dim_feedforward=dim_feedforward,
            dropout=dropout, prediction_length=prediction_length,
        )
        self.learning_rate = learning_rate

    def forward(self, x):
        return self.net(x["encoder_cat"], x["encoder_cont"])

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = torch.sqrt(nn.functional.mse_loss(y_hat, y[0]) + 1e-8)
        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = torch.sqrt(nn.functional.mse_loss(y_hat, y[0]) + 1e-8)
        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)


transformer_params = dict(all_best_params["Vanilla Transformer"])
transformer_final = VanillaTransformerModelFinal(
    cat_sizes=cat_sizes, n_continuous=n_continuous,
    prediction_length=MAX_PREDICTION_LENGTH,
    **transformer_params,
)
recorder_transformer = EpochLossRecorder()
checkpoint_transformer = ModelCheckpoint(
    dirpath="transformer_checkpoints", filename="best-transformer-optuna-{epoch:02d}-{val_loss:.4f}",
    monitor="val_loss", mode="min", save_top_k=1,
)
trainer_transformer = pl.Trainer(
    max_epochs=FINAL_MAX_EPOCHS,
    accelerator=ACCELERATOR, devices=1,
    gradient_clip_val=0.1,
    callbacks=[make_early_stop(), checkpoint_transformer, recorder_transformer],
    logger=False,
    enable_progress_bar=True,
)
trainer_transformer.fit(transformer_final, train_dataloaders=train_dataloader_fixed, val_dataloaders=val_dataloader_fixed)
final_recorders["Vanilla Transformer"] = recorder_transformer
final_models["Vanilla Transformer"] = transformer_final
print("Vanilla Transformer 最佳模型儲存位置：", checkpoint_transformer.best_model_path)

## 12. 繪圖：四模型逐epoch損失曲線（train_loss vs val_loss）

In [ ]:
# ======================= 繪圖：四模型逐epoch損失曲線（train_loss vs val_loss）=======================
for model_name, recorder in final_recorders.items():
    fig, ax = plt.subplots(figsize=(8, 5))
    epochs_train = list(range(1, len(recorder.train_losses) + 1))
    epochs_val = list(range(1, len(recorder.val_losses) + 1))
    ax.plot(epochs_train, recorder.train_losses, marker="o", label="train_loss", color="#4C72B0")
    ax.plot(epochs_val, recorder.val_losses, marker="s", label="val_loss", color="#C44E52")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title(f"{model_name} 訓練損失曲線（最佳超參數，含EarlyStopping）")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fname = f"訓練損失曲線_{model_name.replace(' ', '_')}.png"
    plt.savefig(os.path.join(OUTPUT_DIR, fname), dpi=150)
    plt.close()
    print(f"已輸出 {fname}")

print("\n全部完成！四個最終模型已用最佳超參數重新訓練並存檔，訓練損失曲線也已輸出。")